# ☕ RAG + Emparejador de métodos y cafés (Datasets separados)

Este cuaderno te guía a construir un sistema **RAG (Retrieval-Augmented Generation)** desde cero con dependencias mínimas (TF-IDF + `NearestNeighbors`) y, luego, te muestra cómo llevarlo a librerías populares (Sentence Transformers, FAISS/Chroma, LangChain/llamaindex).

## Objetivos
- Entender **qué es RAG** y cuándo conviene usarlo.
- Implementar un **pipeline minimalista**: *chunking → indexación → recuperación → generación*.
- Crear una **base de conocimiento interesante** (sintética) para consultas.
- Probar consultas con **citas** a las fuentes.
- Ver **rutas de evolución** con librerías de la industria.

Este cuaderno carga dos datasets **separados** —*métodos de preparación* y *cafés*— y un tercer dataset de **emparejamientos** curados.  
Incluye:
- Exploración de datos tabulares.
- Conversión a **documentos** para RAG (TF-IDF minimalista).
- **Q&A** con citas.
- Un pequeño **recomendador** basado en reglas + puntajes de `pairings.csv`.

## ☕ ¿Qué es RAG y cómo lo implementaremos en este notebook?

**RAG (Retrieval-Augmented Generation)** es una técnica que combina **recuperación de información** con **generación de lenguaje natural** para obtener respuestas más precisas y fundamentadas.  
En lugar de depender únicamente del conocimiento interno de un modelo de lenguaje (que puede estar desactualizado o inventar información), **RAG busca primero en una base de conocimiento externa** y luego genera una respuesta utilizando ese contexto como evidencia.

### 🔍 Componentes principales
1. **Base de conocimiento:**  
   Conjunto de documentos o fragmentos de texto que contienen información relevante.  
   En este notebook, esta base estará compuesta por tres fuentes:
   - 🫘 **Cafés:** información sobre orígenes, variedades, procesos y perfiles sensoriales.  
   - ☕ **Métodos de preparación:** parámetros técnicos y características de extracción.  
   - 🔗 **Emparejamientos (“pairings”):** combinaciones óptimas entre café y método con puntuaciones y razones.

2. **Retriever (recuperador):**  
   Dado un texto de consulta (por ejemplo, “¿qué café es ideal para un método V60?”), este componente **busca los fragmentos más similares** en la base de conocimiento.  
   Aquí utilizaremos un **modelo TF-IDF + Nearest Neighbors**, que mide la similitud de las palabras para encontrar los textos más relevantes.

3. **Generator (generador):**  
   A partir de los textos recuperados, se construye una **respuesta contextualizada**.  
   En este caso usaremos un **método extractivo** —seleccionando las oraciones más relevantes y mostrando las citas de sus fuentes—, aunque el mismo pipeline podría extenderse a un LLM (por ejemplo, con LangChain o llama-index).

### 🧩 Cómo se implementa en este notebook
1. Se cargan los datasets `cafes.csv`, `metodos.csv` y `pairings.csv`.  
2. Cada fila se transforma en un documento textual para construir el corpus.  
3. Se entrena un **índice TF-IDF**, que permite comparar la similitud entre consultas y documentos.  
4. Al hacer una pregunta, se:
   - Recuperan los fragmentos más relevantes (`retriever`).
   - Seleccionan las oraciones más informativas (`generator extractivo`).
   - Muestran las **fuentes citadas** para asegurar trazabilidad.

### 🎯 Objetivo final
El propósito es enseñar, de forma práctica y comprensible, cómo crear un pipeline RAG **desde cero**, aplicándolo a un dominio concreto (el café), y cómo este enfoque permite **consultas contextualizadas y recomendaciones basadas en evidencia**.


In [19]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd

cafes = pd.read_csv("/content/drive/MyDrive/2025-2/NLP/cafes_enriquecido.csv")
metodos = pd.read_csv("/content/drive/MyDrive/2025-2/NLP/metodos_enriquecido.csv")
pairings = pd.read_csv("/content/drive/MyDrive/2025-2/NLP/pairings_enriquecido.csv")

print("Cafés:", cafes.shape)
print("Métodos:", metodos.shape)
print("Pairings:", pairings.shape)

Cafés: (20, 9)
Métodos: (12, 8)
Pairings: (240, 4)


In [21]:
cafes.head(3)

,coffee_id,origin,variety,process,altitude_m,roast_level,acidity,body,flavor_notes
0,c_huila_geisha_washed,Huila,Geisha,washed,1750,light,high,light,jazmín; bergamota; miel; florales
1,c_huila_castillo_honey,Huila,Castillo,honey,1650,light-medium,medium,medium,caramelo; durazno; panela
2,c_narino_caturra_washed,Nariño,Caturra,washed,1900,light,high,medium,cítricos; panela; flores


In [22]:
metodos.head(3)

,method_id,method,brew_ratio,grind,water_temp_c,total_time_s,profile_bias,notes
0,m_v60,V60,1:16,medium-fine,92,180,clarity_acidity,"Resalta claridad, florales y acidez brillante."
1,m_kalita,Kalita Wave,1:16,medium-fine,92,185,clarity_balance,Lecho plano; extracción uniforme y balanceada.
2,m_chemex,Chemex,1:17,medium,93,240,clarity_clean,Taza limpia y elegante con poco cuerpo.


In [23]:
pairings.head(3)

,method_id,coffee_id,score,rationale
0,m_v60,c_huila_geisha_washed,5,El método V60 complementa las notas jazmín; be...
1,m_kalita,c_huila_geisha_washed,5,El método Kalita Wave complementa las notas ja...
2,m_chemex,c_huila_geisha_washed,5,El método Chemex complementa las notas jazmín;...


## Conversión a documentos para RAG
Generamos un *corpus* textual para cada **café**, **método** y **pairing** con metadatos.

In [24]:
import pandas as pd

def cafe_doc(row: pd.Series) -> str:
    """
    Genera una descripción textual estandarizada de un café a partir de sus atributos.

    Args:
        row (pd.Series): Fila de un DataFrame que contiene la información de un café.
            Debe incluir al menos las columnas:
            - origin (str): Origen del café.
            - variety (str): Variedad de la planta.
            - process (str): Proceso de beneficio (lavado, honey, natural, etc.).
            - roast_level (str): Nivel de tueste.
            - altitude_m (float | int): Altitud de cultivo en metros.
            - acidity (str): Descripción de la acidez.
            - body (str): Descripción del cuerpo.
            - flavor_notes (str): Notas de sabor.

    Returns:
        str: Cadena formateada con la descripción del café, lista para visualización o
        generación de embeddings textuales.

    Example:
        >>> cafe_doc(df_cafes.iloc[0])
        'CAFÉ | Huila · Caturra · Lavado · medio · alt 1800m. Acidez brillante, cuerpo sedoso. Notas: miel y cítricos.'
    """
    notas = row['flavor_notes']
    return (
        f"CAFÉ | {row.origin} · {row.variety} · {row.process} · {row.roast_level} · "
        f"alt {row.altitude_m}m. Acidez {row.acidity}, cuerpo {row.body}. Notas: {notas}."
    )


def metodo_doc(row: pd.Series) -> str:
    """
    Genera una descripción estandarizada para un método de preparación de café.

    Args:
        row (pd.Series): Fila de un DataFrame con la información del método.
            Debe incluir las columnas:
            - method (str): Nombre del método (p. ej. "V60", "Chemex").
            - brew_ratio (str): Proporción café/agua usada (p. ej. "1:15").
            - grind (str): Nivel de molienda (p. ej. "media", "fina").
            - water_temp_c (float | None): Temperatura del agua en °C.
            - total_time_s (int): Duración total de la preparación en segundos.
            - profile_bias (str): Perfil sensorial o sesgo de extracción.
            - notes (str): Notas adicionales o recomendaciones.

    Returns:
        str: Descripción textual del método, con formato uniforme.

    Example:
        >>> metodo_doc(df_metodos.iloc[0])
        'MÉTODO | V60 · ratio 1:15 · molienda media · 92°C · tiempo 180s. Perfil: balanceado. Notas: extracción limpia.'
    """
    t = row["water_temp_c"]
    temp = f"{int(t)}°C" if pd.notnull(t) else "s/temperatura controlada"
    return (
        f"MÉTODO | {row.method} · ratio {row.brew_ratio} · molienda {row.grind} · "
        f"{temp} · tiempo {int(row.total_time_s)}s. Perfil: {row.profile_bias}. {row.notes}"
    )


def pairing_doc(row: pd.Series, cafes: pd.DataFrame, metodos: pd.DataFrame) -> str:
    """
    Crea una descripción combinada de un emparejamiento (pairing) entre un café y un método de preparación.

    Args:
        row (pd.Series): Fila de un DataFrame de emparejamientos (pairings), que contiene:
            - coffee_id (int): Identificador del café.
            - method_id (int): Identificador del método.
            - score (float): Puntuación de la combinación (por ejemplo, de 1 a 5).
            - rationale (str): Comentario o justificación sensorial.
        cafes (pd.DataFrame): DataFrame con la información de cafés (debe incluir `coffee_id`).
        metodos (pd.DataFrame): DataFrame con la información de métodos (debe incluir `method_id`).

    Returns:
        str: Cadena descriptiva del pairing café × método, con su puntuación y comentarios.

    Example:
        >>> pairing_doc(df_pairings.iloc[0], df_cafes, df_metodos)
        'PAIR | V60 × Huila Caturra (Lavado). Score 4.5/5. Excelente equilibrio entre dulzor y acidez.'
    """
    c = cafes.loc[cafes.coffee_id == row.coffee_id].iloc[0]
    m = metodos.loc[metodos.method_id == row.method_id].iloc[0]
    return (
        f"PAIR | {m.method} × {c.origin} {c.variety} ({c.process}). "
        f"Score {row.score}/5. {row.rationale}"
    )


In [25]:
docs = []
for _, r in cafes.iterrows():
    docs.append({"id": r.coffee_id, "tipo":"cafe", "text": cafe_doc(r)})
for _, r in metodos.iterrows():
    docs.append({"id": r.method_id, "tipo":"metodo", "text": metodo_doc(r)})
for _, r in pairings.iterrows():
    docs.append({"id": f"p_{r.method_id}_{r.coffee_id}", "tipo":"pairing", "text": pairing_doc(r, cafes, metodos)})
len(docs), docs[0], docs[-1]

(272,
 {'id': 'c_huila_geisha_washed',
  'tipo': 'cafe',
  'text': 'CAFÉ | Huila · Geisha · washed · light · alt 1750m. Acidez high, cuerpo light. Notas: jazmín; bergamota; miel; florales.'},
 {'id': 'p_m_pulsar_c_java_anaerobic',
  'tipo': 'pairing',
  'text': 'PAIR | NextLevel Pulsar × Colombia Java (anaerobic). Score 5/5. El método NextLevel Pulsar complementa las notas maracuyá; vino; frutas tropicales del café de Colombia, aportando un perfil high_clarity.'})

## 🔍 Retriever TF-IDF + Nearest Neighbors

Aquí exploramos uno de los enfoques **más clásicos y efectivos para la recuperación de información (IR)**:  
el uso combinado de **TF-IDF (Term Frequency–Inverse Document Frequency)** y **vecinos más cercanos (Nearest Neighbors)**.  
Este método permite encontrar los documentos más relevantes a partir de una consulta textual (*query*), sin requerir modelos neuronales complejos.

---

### 🧠 Objetivo

Construir un **sistema de recuperación semántica** que:
1. Convierte documentos y consultas en vectores numéricos usando **TF-IDF**.
2. Mide la **similitud** entre la consulta y los documentos mediante distancias métricas (por ejemplo, **coseno** o **euclidiana**).
3. Devuelve los **k documentos más cercanos** (los de mayor relevancia semántica).

---

### ⚙️ ¿Qué hace cada componente?

#### 1. **TF-IDF Vectorizer**
- Representa cada texto como un vector en un espacio de dimensión igual al tamaño del vocabulario.  
- Cada componente del vector mide la **importancia de una palabra** dentro del documento y en el corpus total.

Esto permite **resaltar palabras discriminativas** (como “espresso”, “Geisha”, “tueste”) y reducir el peso de las muy comunes (como “el”, “de”, “la”).

---

#### 2. **Nearest Neighbors (k-NN)**
- Se entrena un modelo que guarda los vectores TF-IDF y permite encontrar, para un nuevo texto, los **k vectores más cercanos**.
- La similitud se suele calcular con la **distancia de coseno**:

$$
\text{sim}(A, B) = 1 - \frac{A \cdot B}{\|A\|\|B\|}
$$

donde 1 indica similitud máxima.


In [26]:
from typing import List, Dict, Any
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [27]:
def retrieve(query: str, k: int = 5) -> List[Dict[str, Any]]:
    """
    Recupera los documentos más similares a una consulta textual (`query`)
    usando un modelo de *vectorización* (por ejemplo, TF-IDF o embeddings)
    y un índice de vecinos más cercanos (*Nearest Neighbors*).

    Esta función asume que existen variables globales:
    - `vectorizer`: objeto ajustado (e.g. `TfidfVectorizer` o `SentenceTransformer`)
      con método `.transform()`.
    - `nn`: modelo ajustado de `NearestNeighbors` (de `sklearn.neighbors`).
    - `docs`: lista o DataFrame con los documentos originales asociados a los vectores.

    Args:
        query (str): Texto de la consulta o búsqueda.
        k (int, optional): Número de documentos más similares a recuperar.
            Por defecto es 5.

    Returns:
        List[Dict[str, Any]]: Lista de diccionarios que representan los documentos más similares,
        cada uno con sus metadatos originales y un campo adicional:
        - `'score'` (float): similitud transformada en rango [0, 1],
          calculada como `1 - distancia`.

    Example:
        >>> results = retrieve("café de origen etíope", k=3)
        >>> for r in results:
        ...     print(f"{r['score']:.2f} | {r['title']}")
        0.94 | CAFÉ | Ethiopia · Heirloom · Natural · alt 2100m.
        0.91 | CAFÉ | Kenya · SL28 · Lavado · alt 1900m.
        0.89 | CAFÉ | Colombia · Geisha · Honey · alt 1800m.
    """
    # Vectorizamos la consulta
    qv = vectorizer.transform([query])

    # Buscamos los k vecinos más cercanos
    dist, idx = nn.kneighbors(qv, n_neighbors=k, return_distance=True)

    # Recuperamos los documentos y les asignamos su puntaje de similitud
    out = []
    for d, i in zip(dist[0], idx[0]):
        hit = docs[i].copy()
        hit["score"] = 1 - float(d)
        out.append(hit)

    return out


In [28]:
corpus = [d["text"] for d in docs]
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1,2))
X = vectorizer.fit_transform(corpus)
nn = NearestNeighbors(n_neighbors=6, metric="cosine").fit(X)

## 🧩 Generación Extractiva con Citas

En este capítulo damos un paso más allá del *retriever clásico* para construir un **sistema de respuesta basado en recuperación (RAG: Retrieval-Augmented Generation)**, pero con un enfoque **extractivo**.  
En lugar de “inventar” respuestas, el sistema **selecciona las frases más relevantes** de los documentos recuperados y **las ensambla en una respuesta coherente y citada**.

---

### 🎯 Objetivo

Diseñar un sistema que:
1. Recupera los documentos más relevantes mediante TF-IDF + *Nearest Neighbors*.  
2. Divide esos documentos en **oraciones (sentencize)**.  
3. Calcula la **similitud entre la consulta y cada oración**.  
4. Selecciona las frases más relevantes (hasta un máximo definido).  
5. Genera una **respuesta compuesta** con las oraciones seleccionadas y añade **citas** para mantener trazabilidad.

---

### ⚙️ Flujo general

#### 1. Recuperación inicial
Usa la función `retrieve(query, k)` para obtener los `k` documentos más relevantes respecto a la consulta.  
Cada documento contiene texto completo, metadatos e identificadores.

#### 2. Segmentación en oraciones
Cada texto recuperado se divide en oraciones mediante la función `sentencize()`, usando expresiones regulares que separan tras signos de puntuación (`.`, `!`, `?`).

#### 3. Ranking de oraciones
- Se vectorizan todas las oraciones (`M`) y la consulta (`qv`) con el mismo **vectorizador TF-IDF**.  
- Se calcula la **similitud coseno** entre cada oración y la consulta:

$$
\text{sim}(s_i, q) = \frac{M_i \cdot q}{\|M_i\|\|q\|}
$$

- Las oraciones se ordenan de mayor a menor similitud.

#### 4. Selección de oraciones relevantes
El sistema elige las **top-n oraciones únicas**, evitando repeticiones y manteniendo un límite de contexto (`max_sents`).  
Esto garantiza que la respuesta sea **concisa pero completa**.

#### 5. Generación de la respuesta y citas
Las oraciones seleccionadas se concatenan en un párrafo coherente y se agregan citas breves (`[id · tipo]`) que indican **de qué documento proviene la información**.

---

### 🧠 Ejemplo de resultado

**Pregunta:**  
> ¿Qué métodos resaltan la claridad en taza?

**Respuesta generada:**  
> Los métodos de goteo como V60 y Chemex se caracterizan por ofrecer tazas con gran claridad sensorial y acidez brillante.  
> Estos métodos destacan especialmente en cafés con cuerpo ligero y tueste medio.

**Citas:**  
- [12 · metodo]  
- [4 · cafe]

---

### 📘 Características clave

| Componente | Función | Ventajas |
|-------------|----------|-----------|
| `retrieve()` | Recupera documentos relevantes | Búsqueda rápida basada en TF-IDF |
| `sentencize()` | Divide textos en oraciones | Permite granularidad fina |
| `answer_query()` | Calcula similitud oración-pregunta | Ranking semántico eficiente |
| `ask()` | Interfaz interactiva | Muestra pregunta, respuesta y citas |

---

### 🔍 Diferencias con la generación abstractive

| Aspecto | Generación extractiva | Generación abstractive |
|:--|:--|:--|
| Fuente del texto | Copia oraciones existentes | Genera texto nuevo |
| Verificabilidad | ✅ Alta (mantiene trazabilidad) | ⚠️ Menor (puede inventar datos) |
| Complejidad | Baja | Alta (requiere modelo neuronal) |
| Ejemplo de modelo | TF-IDF + kNN + scoring | GPT, T5, BART, mT5 |

---

### 💡 En resumen

> La **generación extractiva con citas** es una forma simple, interpretable y reproducible de construir un *QA system* basado en documentos.  
> Permite obtener respuestas precisas, trazables y transparentes, ideal para contextos educativos, legales o de investigación, donde **las fuentes deben ser verificables**.


In [29]:
import re
from typing import List, Tuple, Dict, Any
import numpy as np

In [30]:
def sentencize(text: str) -> List[str]:
    """
    Divide un texto en oraciones individuales basándose en signos de puntuación.

    Args:
        text (str): Texto de entrada que se desea fragmentar.

    Returns:
        List[str]: Lista de oraciones extraídas del texto, preservando el orden original.

    Example:
        >>> sentencize("El café es delicioso. Me gusta prepararlo con V60.")
        ['El café es delicioso.', 'Me gusta prepararlo con V60.']
    """
    return re.split(r'(?<=[.!?])\s+', text)


def answer_query(q: str, k: int = 5, max_sents: int = 5) -> Tuple[str, List[str], List[Dict[str, Any]]]:
    """
    Genera una respuesta extractiva a partir de los documentos recuperados mediante la función `retrieve()`.

    Combina el enfoque de **recuperación de contexto (RAG)** con un ranking por similitud
    de oraciones, seleccionando los fragmentos más relevantes para la consulta dada.

    Args:
        q (str): Pregunta o consulta en lenguaje natural.
        k (int, optional): Número de documentos más similares que se recuperan desde `retrieve()`.
            Por defecto es 5.
        max_sents (int, optional): Número máximo de oraciones relevantes que se incluyen en la respuesta.
            Por defecto es 5.

    Returns:
        Tuple[str, List[str], List[Dict[str, Any]]]:
            - **answer** (str): Respuesta compuesta concatenando las oraciones más relevantes.
            - **cits** (List[str]): Lista de citas en formato corto `[id · tipo]` para referencia.
            - **hits** (List[Dict[str, Any]]): Documentos recuperados con sus metadatos y puntajes.

    Example:
        >>> answer, cits, hits = answer_query("¿Qué métodos se usan para preparar café filtrado?")
        >>> print(answer)
        "Los métodos más comunes incluyen V60, Chemex y Kalita Wave..."
        >>> print(cits)
        ['[12 · metodo]', '[7 · cafe]']
    """
    hits = retrieve(q, k)
    sents = []
    for h in hits:
        sents += sentencize(h["text"])

    # Vectorizamos todas las oraciones y la consulta
    M = vectorizer.transform(sents)
    qv = vectorizer.transform([q])

    # Calculamos similitud coseno y ordenamos
    sims = (M @ qv.T).toarray().ravel()
    ord_idx = np.argsort(-sims)

    pick = []
    used = set()
    for i in ord_idx:
        s = sents[i].strip()
        if s and s not in used:
            pick.append(s)
            used.add(s)
        if len(pick) >= max_sents:
            break

    # Respuesta final y citas
    answer = " ".join(pick).strip() or "No hay contexto suficiente."
    cits = [f'[{h["id"]} · {h["tipo"]}]' for h in hits]

    return answer, cits, hits


def ask(q: str, k: int = 5, max_sents: int = 5, show_hits: bool = False) -> Tuple[str, List[str], List[Dict[str, Any]]]:
    """
    Interfaz interactiva para realizar una pregunta al sistema de recuperación extractiva.

    Internamente usa `answer_query()` para generar la respuesta y muestra en consola
    la pregunta, la respuesta sintetizada, las citas y, opcionalmente,
    los documentos recuperados.

    Args:
        q (str): Pregunta o consulta en texto libre.
        k (int, optional): Número de documentos a recuperar. Por defecto 5.
        max_sents (int, optional): Número máximo de oraciones relevantes en la respuesta. Por defecto 5.
        show_hits (bool, optional): Si es True, imprime también los documentos recuperados
            con sus puntajes y tipos. Por defecto False.

    Returns:
        Tuple[str, List[str], List[Dict[str, Any]]]:
            - **ans** (str): Respuesta generada.
            - **cits** (List[str]): Citas breves de los documentos utilizados.
            - **hits** (List[Dict[str, Any]]): Lista de documentos recuperados con sus puntajes.

    Example:
        >>> ask("¿Cuál es la mejor temperatura para preparar un espresso?")
        🧠 Pregunta: ¿Cuál es la mejor temperatura para preparar un espresso?
        📌 Respuesta:
        El espresso se prepara idealmente entre 90°C y 96°C...
        🔎 Citas:
          - [5 · metodo]
    """
    ans, cits, hits = answer_query(q, k, max_sents)
    print("\n🧠 Pregunta:", q)
    print("\n📌 Respuesta:")
    print(ans)
    print("\n🔎 Citas:")
    for c in cits:
        print("  -", c)
    if show_hits:
        print("\n📂 Hits:")
        for h in hits:
            print(f'  • {h["id"]} [{h["score"]:.3f}] ({h["tipo"]})')
    return ans, cits, hits

In [31]:
ask("¿Qué método va mejor con un Geisha de Huila? Dame razones.", show_hits=True)


🧠 Pregunta: ¿Qué método va mejor con un Geisha de Huila? Dame razones.

📌 Respuesta:
PAIR | Chemex × Huila Geisha (washed). PAIR | AeroPress × Huila Geisha (washed). PAIR | Siphon × Huila Geisha (washed). PAIR | V60 × Huila Geisha (washed). PAIR | Espresso × Huila Geisha (washed).

🔎 Citas:
  - [p_m_chemex_c_huila_geisha_washed · pairing]
  - [p_m_aeropress_c_huila_geisha_washed · pairing]
  - [p_m_siphon_c_huila_geisha_washed · pairing]
  - [p_m_v60_c_huila_geisha_washed · pairing]
  - [p_m_espresso_c_huila_geisha_washed · pairing]

📂 Hits:
  • p_m_chemex_c_huila_geisha_washed [0.264] (pairing)
  • p_m_aeropress_c_huila_geisha_washed [0.264] (pairing)
  • p_m_siphon_c_huila_geisha_washed [0.264] (pairing)
  • p_m_v60_c_huila_geisha_washed [0.264] (pairing)
  • p_m_espresso_c_huila_geisha_washed [0.264] (pairing)


('PAIR | Chemex × Huila Geisha (washed). PAIR | AeroPress × Huila Geisha (washed). PAIR | Siphon × Huila Geisha (washed). PAIR | V60 × Huila Geisha (washed). PAIR | Espresso × Huila Geisha (washed).',
 ['[p_m_chemex_c_huila_geisha_washed · pairing]',
  '[p_m_aeropress_c_huila_geisha_washed · pairing]',
  '[p_m_siphon_c_huila_geisha_washed · pairing]',
  '[p_m_v60_c_huila_geisha_washed · pairing]',
  '[p_m_espresso_c_huila_geisha_washed · pairing]'],
 [{'id': 'p_m_chemex_c_huila_geisha_washed',
   'tipo': 'pairing',
   'text': 'PAIR | Chemex × Huila Geisha (washed). Score 5/5. El método Chemex complementa las notas jazmín; bergamota; miel; florales del café de Huila, aportando un perfil clarity_clean.',
   'score': 0.26393203916160823},
  {'id': 'p_m_aeropress_c_huila_geisha_washed',
   'tipo': 'pairing',
   'text': 'PAIR | AeroPress × Huila Geisha (washed). Score 4/5. El método AeroPress complementa las notas jazmín; bergamota; miel; florales del café de Huila, aportando un perfil vers

In [32]:
ask("Quiero un café para Moka con perfil achocolatado y redondo.")


🧠 Pregunta: Quiero un café para Moka con perfil achocolatado y redondo.

📌 Respuesta:
Concentrado y con notas achocolatadas. PAIR | Moka Pot × Huila Geisha (washed). El método Moka Pot complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil intensity. PAIR | Moka Pot × Guatemala Bourbon (washed). El método Moka Pot complementa las notas frambuesa; rosas; panela del café de Huila, aportando un perfil intensity.

🔎 Citas:
  - [m_moka · metodo]
  - [p_m_moka_c_huila_castillo_honey · pairing]
  - [p_m_moka_c_colombia_pink_bourbon · pairing]
  - [p_m_moka_c_huila_geisha_washed · pairing]
  - [p_m_moka_c_guatemala_antigua · pairing]


('Concentrado y con notas achocolatadas. PAIR | Moka Pot × Huila Geisha (washed). El método Moka Pot complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil intensity. PAIR | Moka Pot × Guatemala Bourbon (washed). El método Moka Pot complementa las notas frambuesa; rosas; panela del café de Huila, aportando un perfil intensity.',
 ['[m_moka · metodo]',
  '[p_m_moka_c_huila_castillo_honey · pairing]',
  '[p_m_moka_c_colombia_pink_bourbon · pairing]',
  '[p_m_moka_c_huila_geisha_washed · pairing]',
  '[p_m_moka_c_guatemala_antigua · pairing]'],
 [{'id': 'm_moka',
   'tipo': 'metodo',
   'text': 'MÉTODO | Moka Pot · ratio 1:10 · molienda fine · 95°C · tiempo 300s. Perfil: intensity. Concentrado y con notas achocolatadas.',
   'score': 0.18083408111133237},
  {'id': 'p_m_moka_c_huila_castillo_honey',
   'tipo': 'pairing',
   'text': 'PAIR | Moka Pot × Huila Castillo (honey). Score 3/5. El método Moka Pot complementa las notas caramelo; durazno; panela del caf

In [33]:
ask("¿Sabes de NLP, carros, motos, modelos y computadores?")


🧠 Pregunta: ¿Sabes de NLP, carros, motos, modelos y computadores?

📌 Respuesta:
El método Siphon complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil tea_like. El método AeroPress complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil versatile. El método Espresso complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil concentrated. El método Chemex complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil clarity_clean. El método V60 complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil clarity_acidity.

🔎 Citas:
  - [p_m_siphon_c_huila_castillo_honey · pairing]
  - [p_m_aeropress_c_huila_castillo_honey · pairing]
  - [p_m_chemex_c_huila_castillo_honey · pairing]
  - [p_m_espresso_c_huila_castillo_honey · pairing]
  - [p_m_v60_c_huila_castillo_honey · pairing]


('El método Siphon complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil tea_like. El método AeroPress complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil versatile. El método Espresso complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil concentrated. El método Chemex complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil clarity_clean. El método V60 complementa las notas caramelo; durazno; panela del café de Huila, aportando un perfil clarity_acidity.',
 ['[p_m_siphon_c_huila_castillo_honey · pairing]',
  '[p_m_aeropress_c_huila_castillo_honey · pairing]',
  '[p_m_chemex_c_huila_castillo_honey · pairing]',
  '[p_m_espresso_c_huila_castillo_honey · pairing]',
  '[p_m_v60_c_huila_castillo_honey · pairing]'],
 [{'id': 'p_m_siphon_c_huila_castillo_honey',
   'tipo': 'pairing',
   'text': 'PAIR | Siphon × Huila Castillo (honey). Score 3/5. El método Siphon c

El sistema no entiende el contenido semántico global de la pregunta, solo compara **palabras comunes en un espacio restringido**.  
Por tanto, fuera de su dominio (café), las respuestas serán irrelevantes o aleatorias.

Esto se denomina **problema de *out-of-distribution*** (fuera del dominio de entrenamiento).

🧠 Cómo se puede mitigar

| Estrategia | Descripción |
|-------------|-------------|
| **Filtrado por vocabulario** | Detectar si la consulta contiene menos del 5 % de palabras conocidas y devolver “Consulta fuera de dominio”. |
| **Expansión semántica** | Usar embeddings de modelos multilingües (por ejemplo, `sentence-transformers`) para capturar significado aunque cambien las palabras. |
| **Entrenamiento mixto** | Ampliar el corpus con textos de otros dominios (NLP, tecnología, autos, etc.) para cubrir más vocabulario. |
| **Detección de confianza** | Si la similitud promedio < 0.1, devolver un mensaje como “No tengo información sobre ese tema”. |

---

✅ En resumen

> El sistema respondió sobre café porque **ese es el único tema que “entiende”**.  
> Un *retriever* TF-IDF solo funciona bien dentro del **dominio lingüístico y temático** del corpus que le diste.  
> Fuera de ese dominio, su comportamiento es impredecible, aunque no erróneo desde el punto de vista técnico.

## 🤝 Recomendador de Emparejamientos (Pairings)

Construimos un **sistema de recomendación** que sugiere combinaciones óptimas entre **cafés** y **métodos de preparación**, integrando datos sensoriales y valoraciones previas.  
El objetivo es encontrar los *pairings* (emparejamientos) más compatibles según las características de cada elemento.

---

### 🎯 Objetivo

Diseñar un recomendador que:
1. Calcule la **afinidad sensorial** entre un café y un método.
2. Combine esa afinidad con **puntuaciones históricas** (de catas o usuarios).
3. Devuelva las mejores recomendaciones, ya sea:
   - los **métodos ideales para un café**, o  
   - los **cafés ideales para un método**.

---

### ⚙️ Componentes principales

#### 1. `compatibility_boost(method_row, coffee_row)`
Esta función estima un **"bonus de compatibilidad"** entre un método y un café según su perfil sensorial.

Ejemplos de reglas:
- Si el método tiene sesgo hacia la *claridad* y el café es de **alta acidez y cuerpo ligero**, se suman +0.5 puntos.
- Si el método busca *cuerpo e intensidad* y el café tiene cuerpo alto o notas de **cacao / nuez / chocolate**, se suman +0.5.
- Si el método busca *dulzor y baja acidez* y el café es **natural o con notas frutales**, se suman +0.5.

📈 Cada regla aporta un **boost parcial**, generando un valor final entre 0.0 y 1.5.  
Esto traduce las preferencias sensoriales en **ajustes numéricos interpretables**.

---

#### 2. `recommend_methods_for_coffee(coffee_id, top_n=5)`
Recomienda los **métodos más compatibles** para un café específico.

Proceso:
1. Se obtiene la media de las valoraciones históricas (`pairings`) entre ese café y cada método.
2. Para cada método existente:
   - Se toma la puntuación media (*prior*).
   - Se añade el `compatibility_boost()` correspondiente.
3. Se ordenan los resultados y se devuelven los mejores `top_n`.

Fórmula general:

$$
\text{score}_{c,m} = \text{prior}_{c,m} + \text{boost}(c,m)
$$

donde  
- $ \text{prior}_{c,m} $ = puntuación media histórica (por ejemplo, 2.5 si no hay datos)  
- $ \text{boost}(c,m) $ = ajuste sensorial calculado por reglas.

---

#### 3. `recommend_coffees_for_method(method_id, top_n=5)`
Funciona de manera análoga, pero a la inversa: recomienda **cafés** para un método concreto.

1. Calcula la media histórica de `pairings` para ese método.  
2. Evalúa cada café según su compatibilidad sensorial con el método.  
3. Devuelve los cafés más afines, ordenados por `score`.

---

### 🧠 Ejemplo ilustrativo

**Café:** Huila Geisha (lavado, acidez alta, cuerpo ligero)  
**Método candidato:** Chemex (sesgo “clarity”)  

| Fuente | Valor |
|:-------|:------|
| Puntuación histórica (Chemex × Geisha) | 4.2 |
| Boost por compatibilidad sensorial | +0.5 |
| **Score final** | **4.7** |

➡️ Resultado: Chemex es una excelente recomendación para ese café.

---

### 🔎 Tipo de recomendador

Este sistema es un **modelo híbrido**:
- 🔹 **Colaborativo**, porque usa datos históricos de `pairings` (preferencias reales).  
- 🔹 **Basado en contenido**, porque ajusta el resultado con reglas derivadas de los atributos de cafés y métodos.

Así combina la **experiencia de los catadores** con **conocimiento sensorial experto**.

---

### 💡 En resumen

> El recomendador de emparejamientos predice las combinaciones café–método más compatibles  
> usando una mezcla de **aprendizaje basado en datos** y **razonamiento sensorial explícito**.  
> Es un ejemplo de *sistema híbrido interpretable* que puede adaptarse fácilmente a otros dominios:
> vinos–comidas, autos–motores, música–estado de ánimo, etc.


In [34]:
def compatibility_boost(method_row: pd.Series, coffee_row: pd.Series) -> float:
    """
    Calcula un factor de ajuste ("boost") de compatibilidad entre un método de preparación
    y un café, basado en coincidencias entre el perfil sensorial del método y las
    características del café.

    Args:
        method_row (pd.Series): Fila del DataFrame de métodos.
            Debe incluir al menos la columna:
            - profile_bias (str): Tendencia del método (p. ej. "clarity", "body_intensity", "sweet_low_acidity").
        coffee_row (pd.Series): Fila del DataFrame de cafés.
            Debe incluir las columnas:
            - acidity (str): Nivel de acidez ("high", "medium", "low").
            - body (str): Nivel de cuerpo ("light", "medium", "high").
            - process (str): Proceso de beneficio ("natural", "honey", "washed", etc.).
            - flavor_notes (str): Descripción de notas de sabor.

    Returns:
        float: Valor entre 0.0 y 1.5 (en incrementos de 0.5) que indica el grado de compatibilidad
        entre el método y el café. Un valor mayor sugiere mayor afinidad.

    Example:
        >>> boost = compatibility_boost(metodos.iloc[0], cafes.iloc[2])
        >>> print(boost)
        0.5
    """
    bias = method_row.profile_bias
    boost = 0.0
    notes = coffee_row.flavor_notes.lower()

    if bias.startswith("clarity") and coffee_row.acidity == "high" and coffee_row.body == "light":
        boost += 0.5

    if ("body" in bias or "intensity" in bias) and (
        coffee_row.body == "high" or any(k in notes for k in ["cacao", "nuez", "chocolate"])
    ):
        boost += 0.5

    if "sweet_low_acidity" in bias and (
        coffee_row.process == "natural" or any(k in notes for k in ["frutos", "tropical", "mango", "melón"])
    ):
        boost += 0.5

    return boost


def recommend_methods_for_coffee(coffee_id: Any, top_n: int = 5) -> pd.DataFrame:
    """
    Recomienda métodos de preparación para un café específico, considerando la compatibilidad
    sensorial entre ambos y las valoraciones históricas registradas en `pairings`.

    Args:
        coffee_id (Any): Identificador único del café a evaluar.
        top_n (int, optional): Número máximo de métodos a recomendar. Por defecto 5.

    Globals:
        - cafes (pd.DataFrame): Tabla de cafés con atributos sensoriales.
        - metodos (pd.DataFrame): Tabla de métodos de preparación.
        - pairings (pd.DataFrame): Registros de emparejamientos café–método con puntajes.

    Returns:
        pd.DataFrame: DataFrame con columnas:
            - method_id (int): Identificador del método.
            - method (str): Nombre del método.
            - score (float): Puntuación ajustada por compatibilidad.
        Ordenado de mayor a menor `score`.

    Example:
        >>> recommend_methods_for_coffee(101)
        method_id  method     score
        3          V60        4.8
        2          Chemex     4.5
        1          Aeropress  4.3
    """
    c = cafes.loc[cafes.coffee_id == coffee_id].iloc[0]
    base = pairings[pairings.coffee_id == coffee_id].groupby("method_id")["score"].mean()

    rows = []
    for _, m in metodos.iterrows():
        prior = float(base.get(m.method_id, 2.5))
        score = prior + compatibility_boost(m, c)
        rows.append({
            "method_id": m.method_id,
            "method": m.method,
            "score": round(score, 2)
        })

    recs = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_n)
    return recs


def recommend_coffees_for_method(method_id: Any, top_n: int = 5) -> pd.DataFrame:
    """
    Recomienda cafés para un método de preparación específico, combinando las valoraciones
    promedio de usuarios y el ajuste de compatibilidad calculado con `compatibility_boost()`.

    Args:
        method_id (Any): Identificador del método de preparación.
        top_n (int, optional): Número máximo de cafés a recomendar. Por defecto 5.

    Globals:
        - cafes (pd.DataFrame): Tabla de cafés con atributos sensoriales.
        - metodos (pd.DataFrame): Tabla de métodos.
        - pairings (pd.DataFrame): Registros históricos de emparejamientos con puntajes.

    Returns:
        pd.DataFrame: DataFrame con columnas:
            - coffee_id (int): Identificador del café.
            - origin (str): Origen geográfico del café.
            - variety (str): Variedad botánica.
            - process (str): Proceso de beneficio.
            - score (float): Puntuación ajustada por compatibilidad.
        Ordenado de mayor a menor `score`.

    Example:
        >>> recommend_coffees_for_method(3)
        coffee_id  origin     variety   process   score
        12         Huila      Caturra   Lavado    4.7
        8          Nariño     Castillo  Natural   4.5
    """
    m = metodos.loc[metodos.method_id == method_id].iloc[0]
    base = pairings[pairings.method_id == method_id].groupby("coffee_id")["score"].mean()

    rows = []
    for _, c in cafes.iterrows():
        prior = float(base.get(c.coffee_id, 2.5))
        score = prior + compatibility_boost(m, c)
        rows.append({
            "coffee_id": c.coffee_id,
            "origin": c.origin,
            "variety": c.variety,
            "process": c.process,
            "score": round(score, 2)
        })

    recs = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_n)
    return recs


### Demos rápidos

In [35]:
print("Métodos recomendados para Geisha lavado de Huila:")
recs1 = recommend_methods_for_coffee("c_huila_geisha_washed", top_n=5)
print("Recomendados: métodos para Geisha Huila")
recs1

Métodos recomendados para Geisha lavado de Huila:
Recomendados: métodos para Geisha Huila


,method_id,method,score
0,m_v60,V60,5.5
1,m_kalita,Kalita Wave,5.5
2,m_chemex,Chemex,5.5
10,m_origami,Origami Dripper,5.5
11,m_pulsar,NextLevel Pulsar,5.0


In [36]:
print("Cafés recomendados para V60:")
recs2 = recommend_coffees_for_method("m_v60", top_n=5)
print("Recomendados: cafés para V60")
recs2

Cafés recomendados para V60:
Recomendados: cafés para V60


,coffee_id,origin,variety,process,score
0,c_huila_geisha_washed,Huila,Geisha,washed,5.5
13,c_panama_geisha,Panamá,Geisha,washed,5.5
10,c_colombia_pink_bourbon,Huila,Pink Bourbon,washed,5.5
6,c_etiopia_yirgacheffe,Etiopía,Heirloom,washed,5.5
19,c_java_anaerobic,Colombia,Java,anaerobic,5.0
